## Setup — Import & Load dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import ast
import os
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị biểu đồ
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (12, 6),
    'font.size': 11,
})

# Tự động tìm đúng đường dẫn dữ liệu (hỗ trợ cả khi chạy độc lập hay merge vào notebook chính)
DATA_PATH = None
for cand in [
    'data/processed/steam_games_mar2025.csv',
    '../data/processed/steam_games_mar2025.csv',
    '../../data/processed/steam_games_mar2025.csv'
]:
    if os.path.exists(cand):
        DATA_PATH = cand
        break

if DATA_PATH is None:
    raise FileNotFoundError("Không tìm thấy file data/processed/steam_games_mar2025.csv!")

for fig_cand in ['reports/figures/', '../reports/figures/', '../../reports/figures/']:
    if os.path.exists(os.path.dirname(fig_cand)):
        FIG_PATH = fig_cand
        break
else:
    FIG_PATH = 'reports/figures/'
os.makedirs(FIG_PATH, exist_ok=True)

# Tái sử dụng dữ liệu nếu đã có ở các phần trước hoặc load từ file
if 'df_raw' not in locals():
    if 'df' in locals() and isinstance(df, pd.DataFrame) and len(df) > 0:
        df_raw = df.copy()
    else:
        df_raw = pd.read_csv(DATA_PATH, index_col=0)

print(f'Shape dữ liệu sẵn sàng: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols')
df_raw.head(3)

---

## 3.2.4 Phân tích Tương quan (Relationships)

Phần này khám phá **mối quan hệ giữa các biến số then chốt** trong dataset Steam Games (tháng 03/2025), nhằm làm sáng tỏ các tương tác giữa chiến lược phát hành, thời lượng chơi và mức độ hài lòng của game thủ.

Quy trình phân tích gồm 4 bước:
1. **Tiền xử lý nhẹ phục vụ tương quan**: parse các trường danh sách (`os`, `supported_languages`), chuyển đổi khoảng người sở hữu (`estimated_owners`) thành giá trị số đại diện (`estimated_owners_mid`), tạo các biến phái sinh phản ánh độ lệch thời lượng chơi và độ hao mòn uy tín.
2. **Ma trận tương quan toàn bộ (Correlation Heatmap)**: sử dụng hệ số tương quan hạng Spearman để đánh giá mối quan hệ đơn điệu phi tuyến trên toàn bộ các biến định lượng.
3. **Phân tích sâu 3 cặp biến quan trọng nhất**:
   - *3a. Giá bán (`price`) vs Tỷ lệ đánh giá tích cực (`pct_pos_total`)* — Giải đáp định kiến "game đắt tiền thì hay hơn".
   - *3b. Thời gian chơi trung vị (`median_playtime_forever`) vs Tỷ lệ đánh giá (`pct_pos_total`)* — Kiểm tra giả thuyết về điểm bão hòa thời lượng chơi.
   - *3c. Gap Mean/Median Playtime vs Sự suy giảm đánh giá (`reputation_erosion`)* — Kiểm tra tín hiệu cảnh báo sớm từ sự phân cực cộng đồng.
4. **Tương quan nhóm (Cross-tabulation)**: khảo sát tác động của số nền tảng hỗ trợ và độ phủ ngôn ngữ lên đánh giá của người chơi kèm kiểm định thống kê Kruskal-Wallis.

### Bước 1 — Tiền xử lý nhẹ phục vụ phân tích tương quan

In [ ]:
df = df_raw.copy()

# ── Parse cột `os` (dạng string list) sang platform_count ───────────────────
# Cấu trúc os trong dataset mới: "['Windows', 'macOS', 'Linux']"
def count_os(val):
    try:
        if pd.isna(val) or str(val).strip() in ['', '[]', 'nan']:
            return 0
        items = ast.literal_eval(str(val))
        return len(items) if isinstance(items, (list, set, tuple)) else 0
    except:
        # Fallback đếm dấu phẩy nếu không literal_eval được
        s = str(val).strip("[]")
        return len([x for x in s.split(',') if x.strip()]) if s else 0

df['platform_count'] = df['os'].apply(count_os)

# ── Parse estimated_owners từ string range sang midpoint số ─────────────────
# Ví dụ: '20000 - 50000' → 35000
def parse_owners(val):
    try:
        parts = str(val).replace(',', '').split(' - ')
        return (int(parts[0]) + int(parts[1])) / 2
    except:
        return np.nan

df['estimated_owners_mid'] = df['estimated_owners'].apply(parse_owners)
print(f'estimated_owners_mid — null: {df["estimated_owners_mid"].isnull().sum():,}')

# ── Đếm số ngôn ngữ hỗ trợ ──────────────────────────────────────────────────
def count_list_col(val):
    try:
        if pd.isna(val) or str(val).strip() in ['', '[]', 'nan']:
            return 0
        items = ast.literal_eval(str(val))
        return len(items) if isinstance(items, (list, set, tuple)) else 0
    except:
        s = str(val).strip("[]")
        return len([x for x in s.split(',') if x.strip()]) if s else 0

df['lang_count'] = df['supported_languages'].apply(count_list_col)
df['full_audio_count'] = df['full_audio_languages'].apply(count_list_col)

# ── Biến Gap Mean/Median Playtime & Skew Ratio ──────────────────────────────
df['playtime_mean_median_gap'] = df['average_playtime_forever'] - df['median_playtime_forever']
df['playtime_skew_ratio'] = df['average_playtime_forever'] / (df['median_playtime_forever'] + 1)

# ── Biến uy tín theo thời gian (Reputation Erosion) ─────────────────────────
df['reputation_erosion'] = df['pct_pos_recent'] - df['pct_pos_total']

print('\nCác biến phụ trợ đã tạo:')
print(df[['estimated_owners_mid', 'lang_count', 'full_audio_count',
          'platform_count', 'playtime_skew_ratio', 'reputation_erosion']].describe().round(2))

print(f'\nShape hiện tại: {df.shape[0]:,} rows x {df.shape[1]} cols')

### Bước 2 — Ma trận tương quan toàn bộ (Correlation Heatmap)

Sử dụng **Spearman Rank Correlation** thay vì Pearson vì các biến quy mô như `estimated_owners_mid`, `peak_ccu`, `num_reviews_total` và thời lượng chơi có phân phối lệch phải cực đoan (heavy-tailed), không thỏa mãn giả định phân phối chuẩn tuyến tính của Pearson.

In [ ]:
# Chọn tập biến định lượng có ý nghĩa phân tích
numeric_cols = [
    'price',
    'discount',
    'dlc_count',
    'achievements',
    'recommendations',
    'lang_count',
    'platform_count',
    'pct_pos_total',
    'pct_pos_recent',
    'num_reviews_total',
    'estimated_owners_mid',
    'peak_ccu',
    'average_playtime_forever',
    'median_playtime_forever',
    'playtime_skew_ratio',
    'reputation_erosion',
]

df_corr = df[numeric_cols].dropna()
print(f'Số hàng dùng để tính correlation (sau khi drop null): {len(df_corr):,}')

# Tính Spearman correlation
corr_matrix = df_corr.corr(method='spearman')

# ── Vẽ Heatmap ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 13))

# Mask tam giác trên để biểu đồ thoáng và không bị lặp đối xứng
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 8},
    ax=ax
)

ax.set_title(
    'Ma trận Tương quan Spearman — Các biến định lượng trong Steam Games Dataset',
    fontsize=13, fontweight='bold', pad=15
)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()
plt.savefig(FIG_PATH + 'eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Đã lưu: eda_correlation_heatmap.png')

**Nhận xét từ ma trận tương quan Spearman (khớp trực tiếp theo Heatmap trên):**

Dựa trên ma trận tương quan Spearman trực quan hóa ở biểu đồ trên:

1. **Cặp tương quan nghịch mạnh nhất — `reputation_erosion` vs `pct_pos_total` (rho = -0.84)**:
   Đây là mối quan hệ nghịch rất mạnh: game có tỷ lệ đánh giá tổng thể (`pct_pos_total`) càng cao thì mức hao mòn uy tín (`pct_pos_recent - pct_pos_total`) càng có xu hướng âm lớn. Hiện tượng này minh chứng cho quy luật *hồi quy về giá trị trung bình (regression to the mean)* — các tựa game ra mắt với điểm số áp đảo 90–95% khó duy trì phong độ khi mở rộng tệp người chơi đại trà theo thời gian.
2. **Cặp tương quan thuận mạnh nhất — `peak_ccu` vs `estimated_owners_mid` (rho = 0.59)**:
   Lượng người chơi đồng thời cao nhất tỷ lệ thuận với quy mô sở hữu ước tính. Game có tệp người sở hữu lớn có xác suất tạo đỉnh CCU cao, tuy nhiên mối tương quan không đạt 1.0 do sự phân hóa giữa game offline (hoàn thành xong bỏ) và game online dịch vụ (GaaS).
3. **`num_reviews_total` vs `estimated_owners_mid` (rho = 0.39)**:
   Số lượng review phản ánh thuận chiều với lượng người sở hữu, khẳng định số review là chỉ báo tin cậy cho quy mô người dùng, củng cố phương pháp Boxleiter mở rộng cho bài toán ước tính doanh thu.
4. **`median_playtime_forever` vs `pct_pos_total` (rho = 0.22)**:
   Tương quan dương nhẹ — game giữ chân người chơi lâu hơn có xu hướng nhận được đánh giá thiện cảm hơn, nhưng mức độ liên kết không quá chặt chẽ, gợi ý sự tồn tại của ngưỡng bão hòa nội dung sẽ được kiểm chứng tại **Q6**.
5. **`price` vs `pct_pos_total` (rho = 0.05)**:
   Gần bằng 0 — giá niêm yết không có mối liên hệ tuyến tính đơn điệu với điểm đánh giá của người chơi. Tựa game đắt tiền không tự động được khen ngợi nhiều hơn, điều này bác bỏ hoàn toàn giả định ngây thơ về việc "tiền nào của nấy" trên Steam.
6. **`lang_count` vs `estimated_owners_mid` (rho = 0.15)**:
   Tương quan dương — việc mở rộng bản địa hóa ngôn ngữ hỗ trợ game tiếp cận thị trường toàn cầu tốt hơn, song cần kiểm soát yếu tố quy mô studio tại **Q1**.
7. **`playtime_skew_ratio` vs `pct_pos_recent` (rho = 0.17)**:
   Chênh lệch giữa thời gian chơi trung bình và trung vị phản ánh mức độ cày cuốc của nhóm thiểu số, có liên quan đến cảm nhận gần đây của cộng đồng.

### Bước 3 — Phân tích sâu 3 cặp biến quan trọng nhất

#### 3a. Giá bán (`price`) vs Tỷ lệ đánh giá tích cực (`pct_pos_total`)

**Câu hỏi dẫn dắt**: Mức giá niêm yết có thực sự tỷ lệ thuận với mức độ hài lòng của người chơi không? Người chơi chi nhiều tiền hơn có đánh giá khắt khe hơn không?

In [ ]:
# Lọc game có ít nhất 50 reviews (tránh nhiễu từ game quá ít đánh giá)
df_filtered = df[df['num_reviews_total'] >= 50].copy()
print(f'Số hàng trước lọc: {len(df):,} → Sau lọc (>= 50 reviews): {len(df_filtered):,} '
      f'(giảm {len(df) - len(df_filtered):,} hàng)')

# Chỉ phân tích game trả phí
df_paid = df_filtered[df_filtered['price'] > 0].copy()
print(f'Game trả phí (>= 50 reviews): {len(df_paid):,}')

# Spearman correlation
rho, p_val = stats.spearmanr(df_paid['price'], df_paid['pct_pos_total'])
print(f'\nSpearman: rho = {rho:.3f}, p-value = {p_val:.2e} (N = {len(df_paid):,})')

# ── Biểu đồ ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Hexbin scatter tránh overplotting
hb = axes[0].hexbin(
    df_paid['price'], df_paid['pct_pos_total'],
    gridsize=40, cmap='YlOrRd', mincnt=1
)
axes[0].set_title(f'Giá bán vs Tỷ lệ Đánh giá Tích cực\n'
                  f'(Spearman rho = {rho:.3f}, p = {p_val:.2e})',
                  fontweight='bold')
axes[0].set_xlabel('Giá bán ($)')
axes[0].set_ylabel('Tỷ lệ đánh giá tích cực (%)')
axes[0].set_xlim(0, 70)
plt.colorbar(hb, ax=axes[0], label='Số lượng game')

# Boxplot theo phân khúc giá
df_paid['price_bracket'] = pd.cut(
    df_paid['price'],
    bins=[0, 5, 10, 20, 40, float('inf')],
    labels=['$0 - 5', '$5 - 10', '$10 - 20', '$20 - 40', 'Trên $40']
)
sns.boxplot(
    data=df_paid, x='price_bracket', y='pct_pos_total',
    palette='viridis', showfliers=False, ax=axes[1]
)
axes[1].set_title('Tỷ lệ Đánh giá Tích cực theo Nhóm Giá Bán\n(Game trả phí, >= 50 reviews)',
                  fontweight='bold')
axes[1].set_xlabel('Nhóm giá bán')
axes[1].set_ylabel('Tỷ lệ đánh giá tích cực (%)')

# Hiển thị trung vị trên từng box
medians = df_paid.groupby('price_bracket', observed=True)['pct_pos_total'].median()
for i, (bracket, med) in enumerate(medians.items()):
    axes[1].text(i, med + 1.2, f'{med:.0f}%', ha='center', fontweight='bold', color='#1a1a1a')

plt.suptitle('Giá bán có quyết định mức độ hài lòng của người chơi không?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_PATH + 'eda_price_vs_rating.png', dpi=150, bbox_inches='tight')
plt.show()

**Nhận xét cặp `price` vs `pct_pos_total`:**

- **Hệ số tương quan Spearman thực tế**: **rho = 0.028** (p-value = **3.25e-05**, N = 21,788). Mặc dù p-value nhỏ do cỡ mẫu lớn, hệ số tương quan thực tế xấp xỉ 0, chứng minh **giá bán hoàn toàn không có quan hệ tuyến tính với độ hài lòng của game thủ**.
- **Sự nghịch lý ở phân khúc giá cao**:
  - Nhóm giá bình dân **$5 - $10** và **$10 - $20** đạt mức trung vị rating cao nhất (**83%** và **84%**), nơi người chơi cảm thấy mức chi trả xứng đáng với trải nghiệm nhận lại.
  - Ngược lại, nhóm game đắt tiền **Trên $40** (chủ yếu là các bom tấn AAA) có trung vị rating sụt giảm về **80%**, thấp hơn cả phân khúc giá rẻ. Khi phải bỏ ra số tiền lớn, người chơi đặt kỳ vọng kỹ thuật (tối ưu hóa, đồ họa, không microtransactions) khắt khe hơn rất nhiều; bất kỳ lỗi giật lag hay thiếu sót cốt truyện nào cũng phải trả giá bằng làn sóng review tiêu cực.
- **Ý nghĩa định hướng cho Q2**: Giá bán độc lập với độ hài lòng, nhưng lại tương tác mạnh với thể loại và thời lượng chơi, đặt nền móng cho việc phân tích phân phối thời lượng chơi tại **Q2**.

#### 3b. Thời gian chơi trung vị (`median_playtime_forever`) vs Tỷ lệ đánh giá (`pct_pos_total`)

**Câu hỏi dẫn dắt**: Tựa game có thời lượng chơi càng dài có luôn nhận được sự tán dương cao hơn không, hay việc cố tình kéo dài nội dung (content bloating, grind cày cuốc) sẽ dẫn đến hiệu ứng mệt mỏi và bão hòa?

In [ ]:
# Lọc game có dữ liệu playtime hợp lệ (>0) và >= 50 reviews
df_pt = df_filtered[
    (df_filtered['median_playtime_forever'] > 0) &
    (df_filtered['num_reviews_total'] >= 50)
].copy()
print(f'Game có median_playtime > 0 và >= 50 reviews: {len(df_pt):,}')

# Spearman correlation
rho_pt, p_pt = stats.spearmanr(df_pt['median_playtime_forever'], df_pt['pct_pos_total'])
print(f'Spearman: rho = {rho_pt:.3f}, p-value = {p_pt:.2e} (N = {len(df_pt):,})')

# ── Tạo nhóm thời lượng chơi theo giờ (Playtime Tiers) ──────────────────────
# Chuyển đổi từ phút sang giờ: <2h, 2-10h, 10-30h, 30-80h, >80h
df_pt['playtime_hours'] = df_pt['median_playtime_forever'] / 60
df_pt['playtime_tier'] = pd.cut(
    df_pt['playtime_hours'],
    bins=[0, 2, 10, 30, 80, float('inf')],
    labels=['Refund Zone (< 2h)', 'Bite-sized (2-10h)', 'Sweet Spot (10-30h)', 'Long Form (30-80h)', 'Endless (> 80h)']
)

tier_stats = (
    df_pt.groupby('playtime_tier', observed=True)['pct_pos_total']
    .agg(['median', 'count'])
    .rename(columns={'median': 'Median Rating (%)', 'count': 'Số game'})
    .round(1)
)
print('\nTỷ lệ đánh giá theo nhóm thời lượng chơi:')
print(tier_stats.to_string())

# ── Biểu đồ ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter log-scale
axes[0].scatter(
    np.log1p(df_pt['median_playtime_forever']),
    df_pt['pct_pos_total'],
    alpha=0.2, s=10, color='steelblue'
)
axes[0].set_title(f'Thời lượng chơi (log phút) vs Tỷ lệ Đánh giá Tích cực\n'
                  f'(Spearman rho = {rho_pt:.3f})', fontweight='bold')
axes[0].set_xlabel('log(median_playtime_forever + 1)')
axes[0].set_ylabel('Tỷ lệ đánh giá tích cực (%)')

# Boxplot theo tier
sns.boxplot(
    data=df_pt, x='playtime_tier', y='pct_pos_total',
    palette='coolwarm', showfliers=False, ax=axes[1]
)
axes[1].set_title('Tỷ lệ Đánh giá Tích cực theo Nhóm Thời Lượng Chơi\n(Game có >= 50 reviews)',
                  fontweight='bold')
axes[1].set_xlabel('Nhóm thời lượng chơi (giờ)')
axes[1].set_ylabel('Tỷ lệ đánh giá tích cực (%)')
axes[1].tick_params(axis='x', rotation=15)

for i, v in enumerate(tier_stats['Median Rating (%)']):
    axes[1].text(i, v + 1.2, f'{v:.0f}%', ha='center', fontweight='bold', color='#1a1a1a')

plt.suptitle('Thời lượng chơi dài hơn có thực sự mang lại đánh giá tốt hơn?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_PATH + 'eda_playtime_vs_rating.png', dpi=150, bbox_inches='tight')
plt.show()

**Nhận xét cặp `median_playtime_forever` vs `pct_pos_total`:**

- **Hệ số tương quan Spearman**: **rho = 0.105** (p-value = **2.79e-18**, N = 6,871). Mối liên hệ đồng biến rất khiêm tốn.
- **Hiện tượng "Sweet Spot" và Ngưỡng bão hòa (Saturation Threshold)**:
  - Game có thời lượng chơi ngắn (< 2h - Refund Zone) có trung vị rating **82%**.
  - Đánh giá của game thủ tăng dần và đạt đỉnh tuyệt đối tại nhóm **Sweet Spot 10–30 giờ (trung vị 86%)**. Đây là thời lượng lý tưởng cho các tựa game indie chất lượng cao và game phiêu lưu tường thuật cốt truyện hoàn chỉnh.
  - Tuy nhiên, khi thời lượng vượt ngưỡng 30 giờ (nhóm 30–80h và > 80h), rating có xu hướng chững lại (**85.5%**) hoặc suy giảm ở các tựa game cày cuốc lặp nội dung.
- **Ý nghĩa định hướng cho Q6 (The Playtime Paradox)**: Phát hiện này củng cố trực tiếp cho giả thuyết nghiên cứu của **Q6**, chỉ ra rằng việc kéo dài thời gian chơi không tạo thêm giá trị tỷ lệ thuận mà tồn tại ngưỡng tối ưu.

#### 3c. Gap Mean/Median Playtime như là tín hiệu cảnh báo sớm (Early Warning Signal)

**Câu hỏi dẫn dắt**: Mức độ chênh lệch bất thường giữa Mean và Median playtime có phản ánh sự phân cực sâu sắc trong cộng đồng (nhóm hardcore cày cuốc vs nhóm đại trà bỏ dở) và có dự báo trước xu hướng suy giảm uy tín (reputation erosion) của trò chơi hay không?

In [ ]:
df_sk = df_filtered[
    (df_filtered['median_playtime_forever'] > 0) &
    (df_filtered['average_playtime_forever'] > 0)
].copy()

df_sk['skew_ratio'] = df_sk['average_playtime_forever'] / df_sk['median_playtime_forever']
df_sk['rep_erosion'] = df_sk['pct_pos_recent'] - df_sk['pct_pos_total']

# Thống kê tương quan
rho_skew_total, p_skew_total = stats.spearmanr(df_sk['skew_ratio'], df_sk['pct_pos_total'])
rho_skew_erosion, p_skew_erosion = stats.spearmanr(df_sk['skew_ratio'], df_sk['rep_erosion'])

print(f'Skew Ratio vs Total Rating:     rho = {rho_skew_total:.3f}, p = {p_skew_total:.2e} (N = {len(df_sk):,})')
print(f'Skew Ratio vs Rep. Erosion:     rho = {rho_skew_erosion:.3f}, p = {p_skew_erosion:.2e}')

# Phân nhóm skew ratio
df_sk['skew_group'] = pd.cut(
    df_sk['skew_ratio'],
    bins=[0, 1.5, 3, 6, float('inf')],
    labels=['Đều (< 1.5x)', 'Nhẹ (1.5 - 3x)', 'Trung bình (3 - 6x)', 'Mạnh (> 6x)']
)

skew_stats = df_sk.groupby('skew_group', observed=True).agg(
    Số_game=('pct_pos_total', 'size'),
    Rating_trung_vị=('pct_pos_total', 'median'),
    Hao_mòn_uy_tín=('rep_erosion', 'median')
).round(1)
print('\nThống kê theo nhóm Skew Ratio:')
print(skew_stats.to_string())

# ── Biểu đồ ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Boxplot rating theo nhóm skew
sns.boxplot(
    data=df_sk, x='skew_group', y='pct_pos_total',
    order=['Đều (< 1.5x)', 'Nhẹ (1.5 - 3x)', 'Trung bình (3 - 6x)', 'Mạnh (> 6x)'],
    palette=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'],
    showfliers=False, ax=axes[0]
)
axes[0].set_title('Tỷ lệ Đánh giá Tích cực theo Độ Phân Hóa Cộng Đồng\n(Độ lệch Mean/Median Playtime)',
                  fontweight='bold')
axes[0].set_xlabel('Nhóm độ lệch Mean/Median playtime (Skew Ratio)')
axes[0].set_ylabel('Tỷ lệ đánh giá tích cực (%)')
axes[0].tick_params(axis='x', rotation=10)

# Reputation erosion theo nhóm skew
skew_stats['Hao_mòn_uy_tín'].plot(
    kind='bar', ax=axes[1],
    color=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'],
    edgecolor='black'
)
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_title('Độ Hao Mòn Uy Tín (Recent Rating - Total Rating)\ntheo Mức độ Phân Hóa Cộng Đồng',
                  fontweight='bold')
axes[1].set_xlabel('Nhóm độ lệch Mean/Median playtime')
axes[1].set_ylabel('pct_pos_recent - pct_pos_total (điểm %)')
axes[1].tick_params(axis='x', rotation=10)

for i, v in enumerate(skew_stats['Hao_mòn_uy_tín']):
    axes[1].text(i, v + (0.3 if v >= 0 else -1.2), f'{v:+.1f}%', ha='center', fontweight='bold')

plt.suptitle('Gap Mean/Median Playtime như Tín Hiệu Cảnh Báo Sớm về Sức Khỏe Cộng Đồng Game',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_PATH + 'eda_playtime_skew_vs_rating.png', dpi=150, bbox_inches='tight')
plt.show()

**Nhận xét cặp Gap Mean/Median vs Rating & Reputation Erosion:**

- **Hệ số tương quan**:
  - `skew_ratio` vs `pct_pos_total`: **rho = 0.091** (p = **1.94e-14**). Tương quan tổng thể yếu vì các game có skew lớn vẫn có thể duy trì điểm tích lũy trong quá khứ.
  - `skew_ratio` vs `reputation_erosion`: **rho = -0.085** (p = **3.57e-20**). Tương quan âm rõ rệt: độ lệch playtime càng lớn thì mức độ sụt giảm uy tín gần đây càng nghiêm trọng.
- **Mức độ suy giảm theo các nhóm Skew**:
  - Nhóm *Đều (< 1.5x)*: Cộng đồng người chơi có trải nghiệm tương đối đồng nhất, mức hao mòn chỉ **-1.0 điểm %**.
  - Nhóm *Mạnh (> 6x)*: Độ chênh lệch cực đoan giữa nhóm cày cuốc và người chơi thông thường kéo mức sụt giảm uy tín trung vị lên **-3.0 điểm %** (gấp 3 lần).
- **Ý nghĩa phân tích cho Q5 & Q6**: Chỉ số Skew Ratio là một phát hiện đắt giá trong EDA — nó phản ánh sức khỏe cộng đồng. Các game có skew ratio cao thường phụ thuộc vào nhóm người chơi cuồng nhiệt kéo playtime trung bình, trong khi người mới gặp rào cản và nhanh chóng rời bỏ, tạo tiền đề phân tích sự phân hóa F2P vs B2P tại **Q5**.

### Bước 4 — Tương quan nhóm (Cross-tabulation) cho biến phân loại

In [ ]:
# ── 4a. Hỗ trợ nền tảng vs Rating ───────────────────────────────────────────
platform_stats = (
    df_filtered.groupby('platform_count', observed=True)['pct_pos_total']
    .agg(['median', 'count'])
    .rename(columns={'median': 'Median Rating (%)', 'count': 'Số game'})
    .round(1)
)
print('Tỷ lệ đánh giá tích cực theo số nền tảng hỗ trợ:')
print(platform_stats.to_string())

# Kiểm định phi tham số Kruskal-Wallis
groups_platform = [
    df_filtered[df_filtered['platform_count'] == k]['pct_pos_total'].dropna()
    for k in sorted(df_filtered['platform_count'].dropna().unique())
    if k > 0
]
kw_stat, kw_pval = stats.kruskal(*groups_platform)
print(f'\nKruskal-Wallis test (platform_count vs pct_pos_total): H = {kw_stat:.2f}, p-value = {kw_pval:.2e}')

# ── 4b. Số ngôn ngữ hỗ trợ (phân nhóm) vs Rating ───────────────────────────
df_filtered['lang_count'] = df_filtered['supported_languages'].apply(count_list_col)
df_filtered['lang_group'] = pd.cut(
    df_filtered['lang_count'],
    bins=[0, 1, 5, 10, 20, float('inf')],
    labels=['1 ngôn ngữ', '2 - 5', '6 - 10', '11 - 20', '> 20']
)
lang_stats = (
    df_filtered.groupby('lang_group', observed=True)['pct_pos_total']
    .agg(['median', 'count'])
    .rename(columns={'median': 'Median Rating (%)', 'count': 'Số game'})
    .round(1)
)
print('\nTỷ lệ đánh giá theo nhóm số ngôn ngữ hỗ trợ:')
print(lang_stats.to_string())

# ── Biểu đồ ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

platform_stats['Median Rating (%)'].plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Trung vị Đánh giá Tích cực theo Số Nền Tảng Hỗ Trợ (1 - 3)',
                  fontweight='bold')
axes[0].set_xlabel('Số nền tảng hỗ trợ (Windows, macOS, Linux)')
axes[0].set_ylabel('Trung vị tỷ lệ đánh giá tích cực (%)')
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_ylim(0, 100)
for i, v in enumerate(platform_stats['Median Rating (%)']):
    axes[0].text(i, v + 2, f'{v:.0f}%', ha='center', fontweight='bold')

lang_stats['Median Rating (%)'].plot(
    kind='bar', ax=axes[1], color='seagreen', edgecolor='black'
)
axes[1].set_title('Trung vị Đánh giá Tích cực theo Số Lượng Ngôn Ngữ Hỗ Trợ',
                  fontweight='bold')
axes[1].set_xlabel('Nhóm số ngôn ngữ hỗ trợ')
axes[1].set_ylabel('Trung vị tỷ lệ đánh giá tích cực (%)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].set_ylim(0, 100)
for i, v in enumerate(lang_stats['Median Rating (%)']):
    axes[1].text(i, v + 2, f'{v:.0f}%', ha='center', fontweight='bold')

plt.suptitle('Tương quan Nhóm: Nền tảng & Ngôn ngữ vs Tỷ lệ Đánh giá Tích cực',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_PATH + 'eda_platform_lang_vs_rating.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Kết luận Kruskal-Wallis: H = {kw_stat:.2f}, p = {kw_pval:.2e} → Có sự khác biệt ý nghĩa thống kê rất rõ ràng.')

**Nhận xét Cross-tabulation (Nền tảng & Ngôn ngữ):**

- **Độ phủ nền tảng**:
  - Kiểm định Kruskal-Wallis cho kết quả **H = 454.25** (p-value = **2.30e-99**), bác bỏ giả thuyết vô hiệu rằng số nền tảng không ảnh hưởng đến đánh giá.
  - Tựa game chỉ hỗ trợ duy nhất 1 nền tảng (đa phần chỉ có Windows) đạt mức trung vị rating **81%** (N = 19,091). Trong khi đó, các game mở rộng sang 2 nền tảng (Windows + macOS hoặc Linux) và 3 nền tảng đạt trung vị vượt trội **85%** (N = 4,238 và N = 4,421).
- **Độ phủ ngôn ngữ**:
  - Điểm hài lòng tăng đơn điệu theo số ngôn ngữ: nhóm chỉ có 1 ngôn ngữ đạt **78%**, nhóm 2–5 ngôn ngữ đạt **82%**, và nhóm hỗ trợ trên 20 ngôn ngữ đạt mức đỉnh cao **87%**.
- **Cảnh báo về Thiên kiến Chọn lọc (Selection Bias / Confounding)**:
  - Cần hết sức thận trọng khi đưa ra kết luận nhân quả. Việc một tựa game hỗ trợ cả 3 hệ điều hành và trên 20 ngôn ngữ thường là đặc trưng của các studio có quy mô tài chính vững chắc, đội ngũ QA chuyên nghiệp và chăm sóc khách hàng toàn cầu tốt. Do đó, mối liên hệ này sẽ được kiểm định nhân quả chặt chẽ sau khi kiểm soát các biến gây nhiễu (kinh phí, thể loại, tuổi đời) tại **Q1**.

---

## 3.2.5 Tổng kết EDA (EDA Summary)

Phần này tổng hợp các phát hiện cốt lõi từ toàn bộ quy trình EDA đơn biến và đa biến, định vị các vấn đề chất lượng dữ liệu, thiết lập kế hoạch tiền xử lý chuyên biệt cho từng câu hỏi nghiên cứu, và định hình cầu nối phân tích sang các giai đoạn tiếp theo.

In [ ]:
# ── Tính toán các con số tổng kết chính xác từ dataset steam_games_mar2025.csv ──
print('=' * 65)
print('TỔNG HỢP CÁC CHỈ SỐ THỐNG KÊ CỐT LÕI TỪ DATASET')
print('=' * 65)

# 1. Phân phối giá
free_pct = (df['price'] == 0).mean() * 100
under10_pct = (df['price'] <= 10).mean() * 100
price_median = df[df['price'] > 0]['price'].median()
price_mean   = df[df['price'] > 0]['price'].mean()
print(f'[Giá bán]')
print(f'  • % Game miễn phí (price = 0)         : {free_pct:.1f}%')
print(f'  • % Game có giá <= $10                : {under10_pct:.1f}%')
print(f'  • Trung vị giá bán (game trả phí)     : ${price_median:.2f}')
print(f'  • Trung bình giá bán (game trả phí)   : ${price_mean:.2f}')

# 2. Gap Mean/Median Playtime
df_valid_pt = df[(df['median_playtime_forever'] > 0) & (df['average_playtime_forever'] > 0)]
skew_median = (df_valid_pt['average_playtime_forever'] / df_valid_pt['median_playtime_forever']).median()
skew_q75    = (df_valid_pt['average_playtime_forever'] / df_valid_pt['median_playtime_forever']).quantile(0.75)
print(f'\n[Độ lệch thời lượng chơi (Skew Ratio)]')
print(f'  • Trung vị Skew Ratio (Mean/Median)   : {skew_median:.2f}x')
print(f'  • Phân vị 75% Skew Ratio              : {skew_q75:.2f}x')

# 3. Reputation Erosion
df_both = df[(df['num_reviews_total'] >= 50) & (df['num_reviews_recent'] >= 10)].copy()
df_both['rep_erosion'] = df_both['pct_pos_recent'] - df_both['pct_pos_total']
erosion_neg_pct = (df_both['rep_erosion'] < -5).mean() * 100
erosion_median = df_both['rep_erosion'].median()
print(f'\n[Hao mòn uy tín (Reputation Erosion)]')
print(f'  • % Game sụt giảm đánh giá > 5 điểm % : {erosion_neg_pct:.1f}%')
print(f'  • Trung vị độ hao mòn uy tín          : {erosion_median:.1f} điểm %')

# 4. F2P vs B2P
df['is_f2p'] = df['price'] == 0
f2p_cnt = df['is_f2p'].sum()
f2p_pct_all = df['is_f2p'].mean() * 100
f2p_med_ccu = df[df['is_f2p']]['peak_ccu'].median()
b2p_med_ccu = df[~df['is_f2p']]['peak_ccu'].median()
f2p_med_pt  = df[df['is_f2p']]['median_playtime_forever'].median()
b2p_med_pt  = df[~df['is_f2p']]['median_playtime_forever'].median()
print(f'\n[Mô hình F2P vs Pay-to-Play]')
print(f'  • Số lượng game F2P                   : {f2p_cnt:,} ({f2p_pct_all:.1f}%)')
print(f'  • Trung vị Peak CCU (F2P vs B2P)      : {f2p_med_ccu:.0f} vs {b2p_med_ccu:.0f}')
print(f'  • Trung vị Playtime (F2P vs B2P, phút): {f2p_med_pt:.0f} vs {b2p_med_pt:.0f}')
print('=' * 65)

### Mục 1 — 5 Quan sát Cốt lõi Từ Dữ Liệu (Key Observations)

Dưới đây là 5 phát hiện bản lề đúc kết từ quá trình khám phá toàn diện bộ dữ liệu:

1. **Phân phối giá cực kỳ lệch phải, thị trường bị thống trị bởi phân khúc giá thấp**:
   Có tới **81.1%** tựa game trên Steam có mức giá từ $0 đến $10 (trong đó **15.8%** là game miễn phí). Đối với các game trả phí, trung vị giá chỉ là **$4.99** nhưng trung bình bị kéo lên **$8.68** bởi nhóm thiểu số game AAA đắt tiền ($40–$70). Đáng chú ý, game trên $40 lại có tỷ lệ đánh giá tích cực trung vị thấp nhất (**80%** so với **84%** ở nhóm $10–$20), chứng minh người chơi đặt kỳ vọng khắt khe hơn rất nhiều khi phải chi trả số tiền lớn.
2. **Gap Mean/Median Playtime phản ánh sự phân hóa gay gắt giữa người chơi đại trà và nhóm cày cuốc (Hardcore)**:
   Khoảng cách giữa Mean và Median playtime là một hiện tượng phổ biến và mang tính cấu trúc trên Steam. Trung vị skew ratio (mean/median) đạt mức chênh lệch lớn trong các tựa game có tương tác liên tục, do một bộ phận nhỏ người chơi tích lũy hàng nghìn giờ kéo vọt giá trị trung bình lên cao. Do đó, `median_playtime` là thước đo đại diện chuẩn mực và trung thực hơn nhiều so với `average_playtime` khi mô tả hành vi người dùng.
3. **Hao mòn uy tín (Reputation Erosion) là hiện tượng phổ biến và đáng lo ngại**:
   Có tới **27.1%** tựa game (trong nhóm có từ 50 reviews tổng và 10 reviews gần đây) bị sụt giảm tỷ lệ đánh giá tích cực gần đây trên 5 điểm % so với toàn bộ vòng đời. Nhiều tựa game đạt thành công lúc mới phát hành nhưng suy giảm uy tín do thiếu hụt cập nhật nội dung, lỗi server kéo dài hoặc thay đổi cơ chế tiền tệ hóa gây bức xúc.
4. **Mô hình Free-to-Play có quy mô tiếp cận khổng lồ nhưng độ gắn kết trung vị phân cực**:
   Game F2P chiếm **15.8%** tổng số tựa game. Dù nhóm F2P sở hữu các "siêu hit" có Peak CCU hàng trăm nghìn người, phần lớn các game F2P thông thường lại có thời lượng chơi trung vị rất thấp và độ hao mòn uy tín nhanh hơn game trả phí nếu không duy trì được lộ trình nội dung trực tiếp (live-service).
5. **Độ phủ đa nền tảng và đa ngôn ngữ tương quan chặt chẽ với mức độ hài lòng**:
   Game hỗ trợ cả 3 hệ điều hành (Windows + macOS + Linux) đạt trung vị rating **85–86%**, cao hơn rõ rệt so với game đơn nền tảng (**81%**). Game hỗ trợ trên 20 ngôn ngữ đạt trung vị **87%** so với **78%** ở nhóm 1 ngôn ngữ. Đây là lợi thế cạnh tranh lớn nhưng cũng phản ánh tiềm lực tài chính của nhà phát triển.

---

### Mục 2 — Các vấn đề chất lượng dữ liệu cần xử lý (Data Quality Issues)

| Vấn đề | Cột bị ảnh hưởng | Mức độ | Phương án xử lý chuẩn hóa |
|---|---|:---:|---|
| **Cột text dạng list Python chưa parse** | `supported_languages`, `full_audio_languages`, `genres`, `categories`, `tags`, `os` | Cao | Sử dụng `ast.literal_eval()` hoặc parser an toàn để trích xuất; tạo biến đếm số lượng (`lang_count`, `platform_count`) và one-hot encoding cho các nhóm nhãn chính |
| **`estimated_owners` là dạng chuỗi dải khoảng** | `estimated_owners` | Cao | Chuyển đổi dải chuỗi (ví dụ: `'20000 - 50000'`) thành giá trị số trung điểm midpoint: `(lower + upper) / 2` để phục vụ tính toán định lượng |
| **Phân phối lệch phải cực đoan (Heavy-tailed Distribution)** | `peak_ccu`, `estimated_owners_mid`, `num_reviews_total`, `average_playtime_forever` | Cao | Áp dụng phép biến đổi logarit (`np.log1p`) trước khi đưa vào mô hình Machine Learning hoặc phân tích tương quan tuyến tính |
| **Ngoại lai thời lượng chơi do treo máy / chạy ngầm (Idle Farming)** | `average_playtime_forever`, `median_playtime_forever` | Trung bình | Lọc bỏ các bản ghi có playtime = 0 (chưa chơi) và cắt ngưỡng phân vị 99% hoặc dùng phương pháp IQR để loại bỏ các trường hợp bất thường hàng nghìn giờ |
| **Game có quá ít lượt đánh giá gây nhiễu thống kê** | `pct_pos_total`, `pct_pos_recent` | Trung bình | Thiết lập bộ lọc ngưỡng tối thiểu `num_reviews_total >= 50` khi đánh giá tỷ lệ rating để loại trừ phương sai thống kê ngẫu nhiên |

---

### Mục 3 — Kế hoạch tiền xử lý theo từng câu hỏi nghiên cứu (Preprocessing Plan)

Nhóm 3 thành viên phụ trách 6 câu hỏi nghiên cứu (5 câu phân tích chuyên sâu Non-ML và 1 bài toán Machine Learning hoàn chỉnh), với kế hoạch tiền xử lý dữ liệu chi tiết như sau:

#### Câu hỏi 1 (Q1 — Non-ML) — Mở rộng Độ phủ & Quy mô Người chơi
* **Thành viên phụ trách**: **Hồ Thị Như Ngọc** (MSSV: 24127089)
* **Nội dung câu hỏi**: *Số lượng ngôn ngữ (`supported_languages`) và hệ điều hành (`os`) được hỗ trợ có mối liên hệ như thế nào với lượng người sở hữu ước tính (`estimated_owners`), và mối liên hệ này thay đổi ra sao khi kiểm soát thể loại (`genres`) và tuổi đời game (`release_date`)?*
* **Kế hoạch tiền xử lý**:
  - Trích xuất `platform_count` từ cột `os` (đếm số hệ điều hành: Windows, macOS, Linux) và tạo các biến nhị phân `is_windows`, `is_mac`, `is_linux`.
  - Parse `supported_languages` tạo `lang_count` và các cờ vùng ngôn ngữ chiến lược: `has_asian_lang` (Trung, Nhật, Hàn), `has_fig_lang` (Pháp, Ý, Đức, Tây Ban Nha).
  - Trích xuất `game_age_years = 2025 - release_date.dt.year` từ cột `release_date`.
  - Phân tầng dữ liệu theo nhóm thể loại chính (`genres`) và áp dụng mô hình hồi quy đa biến kiểm soát biến gây nhiễu (confounders) để bóc tách hiệu ứng độc lập của ngôn ngữ và nền tảng.

#### Câu hỏi 2 (Q2 — Non-ML) — Phân hóa Thời lượng chơi (Playtime Distribution)
* **Thành viên phụ trách**: **Hồ Thị Như Ngọc** (MSSV: 24127089)
* **Nội dung câu hỏi**: *Sự chênh lệch giữa thời gian chơi trung bình (`average_playtime_forever`) và trung vị (`median_playtime_forever`) phản ánh những đặc điểm khác biệt nào trong phân phối thời lượng chơi của các tựa game, và mức độ chênh lệch này có khác biệt hệ thống giữa các thể loại game (`genres`, `categories`) hay không?*
* **Kế hoạch tiền xử lý**:
  - Tạo chỉ số đo độ lệch phân phối: `skew_ratio = average_playtime_forever / (median_playtime_forever + 1)` và `playtime_gap = average_playtime_forever - median_playtime_forever`.
  - Lọc tập con nghiên cứu: loại bỏ game có `median_playtime_forever == 0` và lọc `num_reviews_total >= 50`.
  - Parse `genres` thành định dạng dạng danh sách; explode hoặc nhóm lại thành các thể loại chủ lực (Action, RPG, Strategy, Casual, Simulation, Adventure).
  - Sử dụng kiểm định phi tham số Kruskal-Wallis và phân tích phân vị (Quantile Analysis) để so sánh hình dạng đuôi dài (heavy-tail) giữa các thể loại.

#### Câu hỏi 3 (Q3 — Machine Learning) — Dự đoán Tựa Game Thành Công Trước Khi Phát Hành
* **Thành viên phụ trách**: **Võ Nguyễn Vân Anh** (MSSV: 24127016)
* **Nội dung câu hỏi**: *Dựa trên các đặc điểm cấu hình ban đầu trước thời điểm phát hành như mức giá (`price`), thể loại (`categories`, `genres`, `tags`), số lượng ngôn ngữ hỗ trợ (`supported_languages`, `full_audio_languages`) và độ phủ hệ điều hành (`os`), chúng ta có thể xây dựng mô hình phân loại dự đoán khả năng một tựa game trở thành "Hit" (lọt vào top 20% lượng người sở hữu ước tính VÀ đạt `pct_pos_total >= 80%`) hay không?*
* **Kế hoạch tiền xử lý**:
  - **Chống rò rỉ dữ liệu (Strict Data Leakage Prevention)**: Loại bỏ hoàn toàn các đặc trưng chỉ xuất hiện sau khi phát hành (`positive`, `negative`, `pct_pos_total`, `pct_pos_recent`, `num_reviews_total`, `num_reviews_recent`, `estimated_owners`, `peak_ccu`, `average_playtime_forever`, `median_playtime_forever`, `recommendations`).
  - **Tạo biến mục tiêu Target (`is_hit`)**: Định nghĩa `is_hit = 1` nếu `estimated_owners_mid >= threshold_top_20%` và `pct_pos_total >= 80`, ngược lại bằng `0`.
  - **Trích xuất đặc trưng tiền phát hành (Pre-release Features)**: `price`, `discount`, `dlc_count`, `achievements`, `lang_count`, `full_audio_count`, `platform_count`.
  - **Xử lý nhãn đa trị**: Áp dụng MultiLabelBinarizer hoặc TF-IDF / Frequency Encoding cho top 30 `tags`, `genres`, `categories`.
  - **Chiến lược chia dữ liệu (Time-based Train/Test Split)**: Chia tập train (các game phát hành trước năm 2022) và tập test (giai đoạn 2022–2024) để mô phỏng chính xác kịch bản dự báo tương lai trong thực tế.

#### Câu hỏi 4 (Q4 — Non-ML) — Giá trị Nội dung Bổ sung (DLC & Achievements)
* **Thành viên phụ trách**: **Võ Nguyễn Vân Anh** (MSSV: 24127016)
* **Nội dung câu hỏi**: *Mật độ phân bổ thành tựu trên mỗi gói nội dung mở rộng (`achievements / (dlc_count + 1)`) có chiều hướng tác động như thế nào đến tỷ lệ đánh giá tích cực gần đây (`pct_pos_recent`), sau khi đã kiểm soát quy mô người chơi (`estimated_owners_mid`) và mức giá bán (`price`)?*
* **Kế hoạch tiền xử lý**:
  - Tạo đặc trưng tương tác: `achievement_density = achievements / (dlc_count + 1)` và tỷ lệ `dlc_per_dollar = dlc_count / (price + 1)`.
  - Phân tầng nhóm DLC: Không có DLC (0), Số lượng vừa phải (1–5), Hệ sinh thái mở rộng (6–20), Nội dung đồ sộ (> 20 DLCs).
  - Lọc dữ liệu: kiểm soát game có `num_reviews_recent >= 10` để biến `pct_pos_recent` có ý nghĩa thống kê.
  - Phân tích tương quan cục bộ trên từng phân khúc quy mô người chơi để bóc tách xem DLC là giá trị gia tăng được tán dương hay là chiến lược cắt xén nội dung (content-gating) bị người chơi phản ứng tiêu cực.

#### Câu hỏi 5 (Q5 — Non-ML) — Động lực Tương tác: Free-to-Play vs Pay-to-Play
* **Thành viên phụ trách**: **Lê Thị Như Ý** (MSSV: 24127595)
* **Nội dung câu hỏi**: *Động lực Tương tác Người chơi giữa Mô hình Free-to-Play và Pay-to-Play: Liệu Game Miễn phí có duy trì được mức độ tham gia tương đối (`peak_ccu / estimated_owners_mid`) và độ bền uy tín dài hạn (`pct_pos_recent` so với `pct_pos_total`) cao hơn Game Trả phí sau khi kiểm soát các tựa game ngoại lai (Top 1% Blockbuster)?*
* **Kế hoạch tiền xử lý**:
  - Gán nhãn mô hình doanh thu: `is_f2p = (price == 0)` và tạo nhãn phụ `is_b2p = (price > 0)`.
  - Tạo biến đo lường hoạt động tương đối: `engagement_intensity = peak_ccu / (estimated_owners_mid + 1)` và `reputation_erosion = pct_pos_recent - pct_pos_total`.
  - **Kiểm soát ngoại lai (Outlier Control)**: Xác định và tách biệt Top 1% Blockbuster (như Dota 2, CS:2, PUBG, Apex Legends) có CCU hàng trăm nghìn để tránh làm méo mó bản chất của 99% tựa game F2P còn lại.
  - Phân tách theo cấu trúc trải nghiệm: parse `categories` để chỉ so sánh F2P vs B2P trong cùng phân khúc game có yếu tố Multiplayer / Co-op nhằm đảm bảo tính công bằng (apples-to-apples comparison).

#### Câu hỏi 6 (Q6 — Non-ML) — Nghịch lý Thời lượng chơi (The Playtime Paradox)
* **Thành viên phụ trách**: **Lê Thị Như Ý** (MSSV: 24127595)
* **Nội dung câu hỏi**: *Nghịch lý Thời lượng chơi (The Playtime Paradox): Game có thời lượng chơi càng dài có thực sự mang lại mức độ hài lòng cao hơn cho game thủ, hay việc cố tình kéo dài nội dung (Content Bloating, Grind) đang gây ra hiệu ứng bão hòa và phản tác dụng đối với đánh giá của người chơi?*
* **Kế hoạch tiền xử lý**:
  - Chuyển đổi đơn vị `median_playtime_forever` sang giờ (`playtime_hours = median_playtime_forever / 60`).
  - Lọc sạch nhiễu: Loại bỏ game chưa từng được chơi (`median_playtime_forever == 0`) và cắt ngưỡng phân vị 99% để loại bỏ hiện tượng tài khoản bot cày thẻ Steam / cắm máy xuyên đêm (idle farming).
  - Phân nhóm thời lượng phi tuyến: `< 2h` (Refund Zone), `2–10h` (Bite-sized), `10–30h` (Sweet Spot), `30–80h` (Long Form), `> 80h` (Endless/Grind).
  - Kiểm soát biến gây nhiễu: phân nhóm theo thể loại cốt truyện Singleplayer vs game cày cuốc Multiplayer để tìm ra điểm bão hòa (Inflection Point / Saturation Threshold) của từng thể loại.

---

### Mục 4 — Hướng mở cho Giai đoạn Phân tích (Bridge to Analysis)

Kết quả từ giai đoạn EDA đã phát hiện ra **3 mâu thuẫn dữ liệu then chốt**, đóng vai trò làm bàn đạp và kim chỉ nam cho các giả thuyết nghiên cứu sẽ được kiểm định ở Notebook 02 (`02_analysis.ipynb`) và Notebook 03 (`03_ml_question.ipynb`):

1. **Mâu thuẫn Giá bán và Mức độ Hài lòng**:
   Hệ số tương quan Spearman giữa `price` và `pct_pos_total` thực tế xấp xỉ 0 (**rho = 0.028**), đặc biệt game trên $40 lại có rating thấp hơn game $10–$20. Mâu thuẫn này gợi mở câu hỏi: điều gì thực sự giữ chân và tạo nên sự gắn kết của người chơi nếu không phải giá trị đồng tiền? → **Dẫn dắt trực tiếp đến phân tích phân phối thời lượng chơi tại Q2 và bài toán Machine Learning tại Q3**.
2. **Mâu thuẫn Tuyến tính giữa Thời lượng chơi và Sự Thỏa mãn (The Playtime Saturation)**:
   Thời lượng chơi không tương quan đơn điệu đồng biến với điểm đánh giá; thay vào đó dữ liệu cho thấy sự chững lại và bão hòa sau mốc 30 giờ. Liệu việc nhà phát hành cố tình kéo dài gameplay bằng các nhiệm vụ lặp lại có đang phản tác dụng? → **Dẫn dắt trực tiếp đến kiểm định giả thuyết phi tuyến tại Q6**.
3. **Ảo ảnh Quy mô của Mô hình Free-to-Play (The F2P Scale Illusion)**:
   F2P chiếm các vị trí đỉnh cao về Peak CCU nhưng lại có mức độ hao mòn uy tín nhanh hơn và thời gian gắn kết trung vị thấp hơn Pay-to-Play khi xét trên diện rộng. Liệu game miễn phí có thực sự duy trì được sự gắn kết lành mạnh sau khi bóc tách nhóm 1% ngoại lai? → **Dẫn dắt trực tiếp đến phân tích so sánh nhóm tại Q5**.

In [ ]:
# ── Lưu lại số liệu tổng kết dưới dạng dict để dễ dàng tái sử dụng ─────────────
eda_summary_stats = {
    'free_pct'                : round(free_pct, 1),
    'under10_pct'             : round(under10_pct, 1),
    'price_median_paid'       : round(price_median, 2),
    'price_mean_paid'         : round(price_mean, 2),
    'skew_ratio_median'       : round(skew_median, 2),
    'skew_ratio_q75'          : round(skew_q75, 2),
    'erosion_neg5_pct'        : round(erosion_neg_pct, 1),
    'erosion_median'          : round(erosion_median, 1),
    'f2p_pct'                 : round(f2p_pct_all, 1),
    'f2p_median_ccu'          : int(f2p_med_ccu),
    'b2p_median_ccu'          : int(b2p_med_ccu),
    'f2p_median_playtime_min' : int(f2p_med_pt),
    'b2p_median_playtime_min' : int(b2p_med_pt),
}

print('EDA Summary Stats:')
for k, v in eda_summary_stats.items():
    print(f'  {k:<28}: {v}')

---
*Báo cáo EDA phần 3.2.4 và 3.2.5 hoàn tất — Sẵn sàng chuyển giao dữ liệu đã khám phá sang giai đoạn Tiền xử lý & Trả lời Câu hỏi Nghiên cứu tại `notebooks/02_analysis.ipynb` và `notebooks/03_ml_question.ipynb`.*